# Semana 04 — Manipulação de Arquivos e Modularização

**Curso:** Análise de Dados com Python — SENAI (Turma T5)
**UC (MSEP):** Manipulação de Dados com Python e SQL (150h) — Bloco 2, Semanas 03 e 04 (última semana do bloco)

Esta semana fecha o Bloco 2: você veste o boné de analista júnior de **People Analytics** do **Grupo Alfa** (empresa fictícia, 30 colaboradores, 4 setores) e trabalha com 3 arquivos reais que o RH te passou — além de aprender a organizar seu código em funções e módulos reutilizáveis.

> Em cada tópico abaixo: **exemplos resolvidos** + **atividades práticas** para você fazer sozinho(a).

---
### 🟢 Abertura — Semana 04: Manipulação de Arquivos e Modularização

Você já versiona código. **Hoje você veste o boné de analista júnior de People Analytics** — em português, algo como "análise de pessoas": a área de uma empresa que usa dados sobre os próprios funcionários (horário de entrada, atrasos, cadastro, e-mails) para tomar decisões, em vez de decidir só no "achismo". É a área que, por exemplo, descobre se um setor inteiro está sempre atrasado, ou se um cadastro está desatualizado.

Você vai ler dados de arquivos reais do Grupo Alfa e organizar seu código em funções reutilizáveis.

**O que você vai aprender hoje:**
- Ler e escrever arquivos CSV, JSON e Excel com Python
- Trabalhar com datas e expressões regulares em cima de dados reais de ponto e e-mail
- Criar funções com parâmetros, valores padrão e `return`, e usar lambda e módulos

### 📌 Antes de começar: os 3 arquivos desta semana

O RH do Grupo Alfa te passou 3 arquivos (nesta mesma pasta):

| Arquivo | Formato | Sistema de origem |
|---|---|---|
| `cadastro_funcionario.csv` | CSV | Sistema de folha de pagamento |
| `emails_corporativos.json` | JSON | Sistema de e-mail corporativo |
| `recursos_humanos.xlsx` | Excel (2 abas) | Sistema de ponto |

**Rodando no Google Colab?** Diferente do VS Code local, o Colab não enxerga os arquivos desta pasta automaticamente — rode a célula abaixo pra enviá-los.

In [1]:
# Se estiver rodando no Google Colab, esta célula abre a janela de upload
# e organiza os arquivos na subpasta dataset/, igual à estrutura do repositório.
# Selecione os 3 arquivos desta semana: cadastro_funcionario.csv, emails_corporativos.json, recursos_humanos.xlsx
import os

try:
    from google.colab import files
    enviados = files.upload()
    os.makedirs("dataset", exist_ok=True)
    for nome in enviados:
        os.replace(nome, f"dataset/{nome}")
except ImportError:
    print("Rodando localmente (VS Code) — os arquivos já estão em dataset/, na pasta desta semana.")

Rodando localmente (VS Code) — os arquivos já estão na mesma pasta do notebook.


💡 **O que é o módulo `os`?** `os` vem de "operating system" (sistema operacional). É um módulo que já vem **dentro do Python** (não precisa instalar, igual `csv`/`json`/`re`) e serve pra seu código conversar com o sistema operacional do computador: criar pasta, mover/renomear arquivo, verificar se um arquivo existe — tudo que você normalmente faria clicando no Explorador de Arquivos do Windows (ou no Finder do Mac), só que via código.

Nesta célula, `os.makedirs("dataset", exist_ok=True)` cria a pasta `dataset/` (o parâmetro `exist_ok=True` evita erro caso ela já exista); e `os.replace(nome, f"dataset/{nome}")`, dentro do `for`, move cada arquivo enviado da raiz (onde o Colab sempre larga o upload) pra dentro da pasta `dataset/`.

---
## 1. Arquivos: CSV, JSON e Excel

Até agora, todo dado que você usou existia só enquanto o programa rodava, ou foi digitado à mão no próprio código. A partir de hoje isso muda: cada um dos 3 arquivos do Grupo Alfa vem de um sistema diferente, por isso está num formato diferente — CSV quando o dado é tabular e simples, JSON quando tem estrutura mais livre, Excel quando vem de planilhas com múltiplas abas.

💡 **O que é uma "biblioteca"?** É um conjunto de ferramentas prontas que alguém já programou pra você não precisar reinventar a roda. `csv`, `json` e `re` já vêm **dentro do Python** — basta dar `import`, sem instalar nada. Já `openpyxl` (que você vai usar daqui a pouco pra ler o Excel) é uma biblioteca **de terceiros**: não vem com o Python, então precisa ser **instalada** antes do primeiro uso — você vai ver como fazer isso logo abaixo, no Exemplo 3. (Mais pra frente, na Seção 4, você vai aprender a criar a sua própria biblioteca — lá ela vai se chamar **módulo**: um módulo nada mais é do que uma biblioteca que você mesmo escreveu.)

### 📖 Antes do primeiro código: o que significa "abrir" um arquivo?

Assim como você abre um arquivo do Windows dando duplo clique nele, o Python também precisa "abrir" um arquivo antes de conseguir ler o que está escrito dentro. E, do mesmo jeito que é boa prática fechar um programa quando termina de usar, o Python também precisa **fechar** o arquivo depois de ler — senão ele fica "reservado", e nenhum outro programa consegue mexer nele enquanto isso. Em Python, isso é feito assim:

```python
with open("cadastro_funcionario.csv", encoding="utf-8") as arquivo:
    # aqui dentro, o arquivo está aberto e disponível
    ...
# aqui fora, o Python já fechou o arquivo sozinho
```

Repare em 3 coisas:
- `open("cadastro_funcionario.csv", encoding="utf-8")` é a parte que efetivamente abre o arquivo. Você já vai entender o `encoding="utf-8"` em instantes.
- `as arquivo` dá um apelido pro arquivo aberto — você poderia chamar de qualquer nome (`f`, `meu_arquivo`...), mas `arquivo` deixa claro o que é.
- A palavra `with` é quem garante que o Python vai **fechar o arquivo automaticamente** assim que o bloco indentado (a parte recuada, embaixo dos dois-pontos) terminar — mesmo se algo der errado no meio do caminho. Sempre que for ler ou escrever um arquivo em Python, você vai começar com `with open(...) as ...:`. Decore essa estrutura: ela vai aparecer o resto do curso inteiro.

### 🔹 Exemplo 1 — Lendo o cadastro de funcionários (CSV)

In [ ]:
import csv

with open("dataset/cadastro_funcionario.csv", encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo)
    funcionarios = list(leitor)

print(f"Total de funcionários: {len(funcionarios)}")
print(funcionarios[0])

### 🔹 Exemplo 2 — Lendo os e-mails corporativos (JSON)

In [ ]:
import json

with open("dataset/emails_corporativos.json", encoding="utf-8") as arquivo:
    emails = json.load(arquivo)

print(emails[0])

> ⚠️ **Veja como é um erro real do Python.** Tente rodar `open("nao_existe.json", "r")` numa célula nova — você vai ver `FileNotFoundError: [Errno 2] No such file or directory: 'nao_existe.json'`. É o erro mais comum de quem trabalha com arquivos (nome digitado errado, ou arquivo que ainda não existe). Você pode se proteger com `try`/`except` — veja a célula abaixo.

In [ ]:
try:
    with open("nao_existe.json", "r", encoding="utf-8") as arquivo:
        dados = json.load(arquivo)
except FileNotFoundError:
    print("Arquivo não encontrado — verifique o nome ou crie o arquivo primeiro.")

> ⚠️ **Cuidado com o `encoding`.** Repare que os dois exemplos acima usam `encoding="utf-8"`. Se você esquecer esse parâmetro no Windows, o Python pode abrir o arquivo com outra codificação padrão do sistema — e palavras acentuadas quebram silenciosamente, sem gerar erro nenhum: `"Logística"` pode virar `"LogÃ­stica"` na tela. É um dos bugs mais chatos de encontrar porque **o programa não trava, só mostra o dado errado**.

### 🔹 Exemplo extra — Escrevendo um CSV

Até agora você só **leu** arquivos. Mas o RH também vai pedir pra você **gerar** um novo CSV — por exemplo, uma lista só com os funcionários de um setor. Para escrever, o Python usa `csv.DictWriter`, o "irmão" de escrita do `csv.DictReader` que você já usou:

In [ ]:
import csv

with open("dataset/cadastro_funcionario.csv", encoding="utf-8") as arquivo:
    funcionarios = list(csv.DictReader(arquivo))

funcionarios_estoque = [f for f in funcionarios if f["setor"] == "Estoque"]

with open("funcionarios_estoque.csv", "w", encoding="utf-8", newline="") as arquivo:
    escritor = csv.DictWriter(arquivo, fieldnames=funcionarios[0].keys())
    escritor.writeheader()
    escritor.writerows(funcionarios_estoque)

print(f"{len(funcionarios_estoque)} funcionários do Estoque salvos em funcionarios_estoque.csv")

💡 **O que aconteceu em cada linha:**
- `"w"` no `open(...)` significa **escrita** (`write`) — diferente do modo de leitura que você já usa. Cuidado: `"w"` **sobrescreve** o arquivo inteiro se ele já existir.
- `newline=""` é importante ao escrever CSV, principalmente **no Windows (VS Code local)**: sem ele, o módulo `csv` pode inserir uma linha em branco entre cada registro — um bug clássico de Windows. No Colab (que roda em Linux) isso costuma não aparecer, mas escrever sempre com `newline=""` evita o problema nos dois ambientes.
- `csv.DictWriter(arquivo, fieldnames=...)` precisa saber, de antemão, quais são as colunas — por isso passamos `funcionarios[0].keys()`, as mesmas chaves que já vieram do CSV original.
- `.writeheader()` escreve a primeira linha (os nomes das colunas); `.writerows(...)` escreve o restante, um dicionário por linha.

### 🔹 Exemplo 3 — Lendo o banco de horas (Excel, múltiplas abas)

📦 **Antes do código: instalando o `openpyxl`.** Diferente de `csv`, `json`, `re` e `datetime` (que já vêm dentro do Python), `openpyxl` é uma biblioteca **de terceiros** — feita por outras pessoas, não pelo Python — e por isso precisa ser **instalada** antes do primeiro uso. Rode a célula abaixo uma vez: ela funciona tanto no Colab quanto no VS Code local, e se o `openpyxl` já estiver instalado, ela simplesmente não faz nada (não tem problema rodar de novo).

> ⚠️ **Se você pular esta célula**, ao rodar o próximo bloco de código vai aparecer `ModuleNotFoundError: No module named 'openpyxl'`. Se isso acontecer, é só voltar aqui e rodar esta célula antes de tentar de novo.

In [ ]:
%pip install -q openpyxl

In [ ]:
from openpyxl import load_workbook

planilha = load_workbook("dataset/recursos_humanos.xlsx")
print("Abas disponíveis:", planilha.sheetnames)

aba_ponto = planilha["banco_horas"]
print(f"Total de registros de ponto: {aba_ponto.max_row - 1}")  # -1 por causa do cabeçalho

for linha in aba_ponto.iter_rows(min_row=2, max_row=4, values_only=True):
    print(linha)

💡 **Os 3 parâmetros de `iter_rows(...)`:**
- `min_row=2` — comece a partir da linha 2 (pulando a linha 1, que é o cabeçalho com os títulos das colunas, não um dado de verdade).
- `max_row=4` — pare na linha 4 (senão o Python tentaria imprimir os 100 registros de uma vez).
- `values_only=True` — devolva só os valores de cada célula (ex.: `'Atendente'`), em vez de um "objeto célula" mais complexo que o Excel usa internamente. Sem esse parâmetro, você teria que escrever `celula.value` pra cada campo — `values_only=True` já entrega o valor pronto.

Não se preocupe com o formato `datetime.datetime(2023, 1, 2, 0, 0)` que aparece na saída — você vai entender exatamente o que é isso na próxima seção.

### ✏️ Atividade 1 — Contar por setor (CSV)

Usando `cadastro_funcionario.csv`, filtre e conte quantos funcionários existem no setor `"Estoque"`.

**✅ Gabarito:**

In [ ]:
import csv

with open("dataset/cadastro_funcionario.csv", encoding="utf-8") as arquivo:
    funcionarios = list(csv.DictReader(arquivo))

funcionarios_estoque = [f for f in funcionarios if f["setor"] == "Estoque"]
print(f"Funcionários no Estoque: {len(funcionarios_estoque)}")

### ✏️ Atividade 2 — Escrever um resumo (JSON)

A partir do CSV, monte um dicionário contando quantos funcionários existem em cada setor e salve esse resumo em um novo arquivo `relatorio_setores.json`.

**✅ Gabarito:**

In [ ]:
import csv, json

with open("dataset/cadastro_funcionario.csv", encoding="utf-8") as arquivo:
    funcionarios = list(csv.DictReader(arquivo))

contagem_setores = {}
for f in funcionarios:
    setor = f["setor"]
    contagem_setores[setor] = contagem_setores.get(setor, 0) + 1

resumo = {"total_funcionarios": len(funcionarios), "por_setor": contagem_setores}

with open("relatorio_setores.json", "w", encoding="utf-8") as arquivo:
    json.dump(resumo, arquivo, ensure_ascii=False, indent=2)

print(resumo)

### ✏️ Atividade 3 — Explorar o Excel

Abra `recursos_humanos.xlsx`, confirme que a aba `banco_horas` tem 100 registros, e imprima os 5 primeiros.

**✅ Gabarito:**

In [ ]:
from openpyxl import load_workbook

planilha = load_workbook("dataset/recursos_humanos.xlsx")
aba_ponto = planilha["banco_horas"]

print(f"Total de registros: {aba_ponto.max_row - 1}")

for linha in aba_ponto.iter_rows(min_row=2, max_row=6, values_only=True):
    print(linha)

> ⚠️ **Atenção antes da próxima atividade.** Repare que, no resultado do Exemplo 1, `'id_funcionario': '1'` aparece **entre aspas** — isso significa que veio como texto, não como número. Isso acontece porque **todo valor lido de um CSV chega como texto**, mesmo que pareça um número. Já no JSON (Exemplo 2), `'id_funcionario': 1` aparece **sem aspas** — esse já é um número de verdade. Isso importa porque, em Python, `"9" == 9` é `False` — texto e número nunca são considerados iguais, mesmo representando "a mesma coisa" pros nossos olhos. Se for comparar o `id_funcionario` do CSV com o do JSON, primeiro transforme o do CSV em número com `int(...)`:
```python
id_do_csv = funcionarios[0]["id_funcionario"]   # '1' (texto)
id_convertido = int(id_do_csv)                    # 1 (número)
```

### ✏️ Atividade 4 — Cruzar duas fontes

Escolha um `id_funcionario`, busque o nome dele no CSV e o e-mail dele no JSON, e imprima os dois juntos (ex.: `"Igor Pereira: igor.pereira@grupoalfa.com.br"`). Lembre-se de converter o id do CSV com `int(...)` antes de comparar.

**✅ Gabarito:**

In [ ]:
import csv, json

with open("dataset/cadastro_funcionario.csv", encoding="utf-8") as arquivo:
    funcionarios = list(csv.DictReader(arquivo))

with open("dataset/emails_corporativos.json", encoding="utf-8") as arquivo:
    emails = json.load(arquivo)

id_escolhido = 9

nome = None
for f in funcionarios:
    if int(f["id_funcionario"]) == id_escolhido:
        nome = f["funcionario"]
        break

email = None
for e in emails:
    if e["id_funcionario"] == id_escolhido:
        email = e["email"]
        break

print(f"{nome}: {email}")

---
✅ **Checagem rápida — antes de avançar:**
Você consegue abrir o `cadastro_funcionario.csv` e contar quantos registros ele tem, sem olhar o exemplo?

---

---
## 2. Datas e Expressões Regulares

> 🔗 **Por que isso agora?** A aba `banco_horas` do Excel tem colunas de data e hora (`datahora_entrada`, `datahora_saida`). Pra calcular quanto cada funcionário trabalhou — ou se chegou atrasado — você precisa saber operar com essas datas. E o arquivo de e-mails tem 5 registros com formato quebrado, que só dá pra encontrar com um padrão de texto: é aí que entra o regex.

### Datas com `datetime`

### 🔹 Exemplo 1 — Calculando horas trabalhadas

💡 **Antes do código: como ler `datetime(2023, 1, 2, 8, 30)`?** Os 5 números são sempre nessa ordem: **ano, mês, dia, hora, minuto**. Aqui: 2023 (ano), 1 (janeiro), 2 (dia 2), 8 (8 horas), 30 (30 minutos) — ou seja, 2 de janeiro de 2023, às 8h30.

In [ ]:
from openpyxl import load_workbook

planilha = load_workbook("dataset/recursos_humanos.xlsx")
aba_ponto = planilha["banco_horas"]

registro = None
for linha in aba_ponto.iter_rows(min_row=2, values_only=True):
    if linha[0] == 1 and linha[3].date().isoformat() == "2023-01-02":
        registro = linha  # Ana Souza (id 1), 02/01/2023
        break

entrada = registro[4]
saida = registro[5]

horas_trabalhadas = (saida - entrada).seconds / 3600
print(f"Horas trabalhadas: {horas_trabalhadas:.2f}h")

💡 **O que aconteceu em cada linha:**
- `entrada` e `saida` vêm direto da planilha `recursos_humanos.xlsx` — a célula abre o arquivo e busca a linha certa (mesmo padrão da Atividade 4), em vez de digitar o horário à mão.
- Quando você subtrai duas datas/horas em Python (`saida - entrada`), o resultado **não é um número** — é um "pacote" especial chamado `timedelta` ("diferença de tempo"), que guarda quantos dias e quantos segundos existem entre as duas datas.
- `.seconds` pega só a parte em segundos desse pacote. Como `entrada` e `saida` são no mesmo dia, isso já nos dá o total certo.
- 1 hora tem 3600 segundos (60 minutos × 60 segundos) — por isso dividimos por `3600`: é a conversão de segundos para horas.
- `:.2f` dentro do `f"..."` significa "mostre só 2 casas depois da vírgula". Sem isso, apareceria algo como `7.916666666666667` — difícil de ler.

### 🔹 Exemplo 2 — Calculando o atraso em minutos

In [ ]:
from openpyxl import load_workbook

planilha = load_workbook("dataset/recursos_humanos.xlsx")
aba_ponto = planilha["banco_horas"]

registro = None
for linha in aba_ponto.iter_rows(min_row=2, values_only=True):
    if linha[0] == 1 and linha[3].date().isoformat() == "2023-01-02":
        registro = linha  # Ana Souza (id 1), 02/01/2023
        break

entrada_prevista_min = 8 * 60  # 08:00, em minutos
entrada_real = registro[4]
entrada_real_min = entrada_real.hour * 60 + entrada_real.minute

atraso_minutos = entrada_real_min - entrada_prevista_min
print(f"Atraso: {atraso_minutos} minutos")

> ⚠️ **Veja como é um erro real do Python.** Tente criar `datetime(2023, 13, 1)` (mês 13 não existe) numa célula nova — você vai ver `ValueError: month must be in 1..12`. Diferente do `FileNotFoundError` (um arquivo que não existe), esse é um erro de **valor inválido**: a data em si não existe no calendário.

### ✏️ Atividade 5 — Outro registro

Usando o Excel, pegue o registro de outro `id_funcionario` do dia 02/01/2023 e calcule as horas trabalhadas dele.

**✅ Gabarito:**

In [ ]:
from openpyxl import load_workbook

planilha = load_workbook("dataset/recursos_humanos.xlsx")
aba_ponto = planilha["banco_horas"]

registro = None
for linha in aba_ponto.iter_rows(min_row=2, values_only=True):
    if linha[0] == 3 and linha[3].date().isoformat() == "2023-01-02":
        registro = linha  # Carlos Mendes (Estoque), id 3, 02/01/2023
        break

entrada = registro[4]
saida = registro[5]

horas_trabalhadas = (saida - entrada).seconds / 3600
print(f"Horas trabalhadas: {horas_trabalhadas:.2f}h")

### ✏️ Atividade 6 — Atraso de outro funcionário

Calcule o atraso (em minutos) do mesmo registro da Atividade 5, comparando com a entrada prevista das 08:00. Se o resultado for negativo, o que isso significa?

**✅ Gabarito:**

In [ ]:
from openpyxl import load_workbook

planilha = load_workbook("dataset/recursos_humanos.xlsx")
aba_ponto = planilha["banco_horas"]

registro = None
for linha in aba_ponto.iter_rows(min_row=2, values_only=True):
    if linha[0] == 3 and linha[3].date().isoformat() == "2023-01-02":
        registro = linha  # Carlos Mendes (Estoque), id 3, 02/01/2023
        break

entrada_prevista_min = 8 * 60
entrada_real = registro[4]
entrada_real_min = entrada_real.hour * 60 + entrada_real.minute

atraso_minutos = entrada_real_min - entrada_prevista_min
print(f"Atraso: {atraso_minutos} minutos")  # negativo = chegou antes do previsto

### Expressões regulares com `re`

Uma expressão regular (regex) é um padrão de texto usado para **validar** ou **buscar** formatos específicos. Você já sabe que 5 dos 30 e-mails de `emails_corporativos.json` vieram quebrados do sistema de TI — é hora de encontrá-los.

### 🔹 Exemplo 1 — Validando um e-mail

💡 **Antes do código: o que é o "r" antes das aspas?** Repare que o padrão vai começar com `r"..."` — esse `r` colado nas aspas cria o que chamamos de *raw string* ("texto bruto"). Padrões de regex usam bastante a barra invertida (`\`), e sem o `r` o Python tentaria entender essa barra de outro jeito, o que quebraria o padrão. **Regra prática: sempre que for escrever um padrão de regex, comece com `r"..."`.**

In [ ]:
import re

email = "ana.souza@grupoalfa.com.br"
padrao = r"^[^\s@]+@[^\s@]+\.[^\s@]+$"

if re.match(padrao, email):
    print("E-mail parece válido!")
else:
    print("E-mail inválido.")

> 📌 **Este exemplo ainda não usou nenhum arquivo.** Você testou o padrão com um e-mail digitado direto no código, só pra entender a lógica do regex antes de aplicar aos dados reais. O próximo exemplo (CPF) também é assim — dado digitado na hora, sem abrir arquivo nenhum. Isso muda no Exemplo 2, que volta a trabalhar com o arquivo `emails_corporativos.json`.

### 🔹 Exemplo extra — Validando um CPF

O mesmo raciocínio do e-mail serve pra validar qualquer formato de texto — só muda o padrão. Um CPF no formato `000.000.000-00` tem uma estrutura mais rígida que um e-mail: sempre 11 dígitos, sempre nas mesmas posições.

In [ ]:
import re

cpf_valido = "123.456.789-00"
cpf_invalido = "123.456.789-0"  # faltou um dígito antes do traço

padrao_cpf = r"^\d{3}\.\d{3}\.\d{3}-\d{2}$"

for cpf in [cpf_valido, cpf_invalido]:
    if re.match(padrao_cpf, cpf):
        print(f"{cpf}: CPF parece válido!")
    else:
        print(f"{cpf}: CPF inválido.")

💡 **`\d{3}` em vez de `[^\s@]+`:** no e-mail, o tamanho do texto antes/depois do `@` varia de pessoa pra pessoa, por isso o padrão usava "1 ou mais caracteres" (`+`). Um CPF tem tamanho fixo — por isso o padrão troca isso por uma contagem exata: `\d` é qualquer dígito (0-9), e `{3}` repete isso 3 vezes seguidas. O `.` e o `-` aparecem literalmente, nas mesmas posições.

### 🔹 Exemplo 2 — Separando todos os e-mails válidos dos inválidos

In [ ]:
import json, re

with open("dataset/emails_corporativos.json", encoding="utf-8") as arquivo:
    emails = json.load(arquivo)

padrao = r"^[^\s@]+@[^\s@]+\.[^\s@]+$"

validos = []
invalidos = []
for registro in emails:
    if re.match(padrao, registro["email"]):
        validos.append(registro)
    else:
        invalidos.append(registro)

print(f"E-mails válidos: {len(validos)}")
print(f"E-mails inválidos: {len(invalidos)}")

### 🔹 Exemplo extra — Listando só os e-mails válidos

Separar em duas listas (Exemplo 2) é o primeiro passo. Agora que você tem `validos` e `invalidos` prontos, dá pra usar cada lista sozinha — aqui, só pra imprimir o e-mail de quem está correto.

In [ ]:
import json, re

with open("dataset/emails_corporativos.json", encoding="utf-8") as arquivo:
    emails = json.load(arquivo)

padrao = r"^[^\s@]+@[^\s@]+\.[^\s@]+$"

validos = [registro for registro in emails if re.match(padrao, registro["email"])]

for registro in validos:
    print(registro["email"])

💡 **Por que abrir o arquivo de novo?** Você já tinha aberto `emails_corporativos.json` no Exemplo 2, mas essa célula abre de novo de propósito — assim ela funciona sozinha, mesmo que você rode só ela depois de reabrir o notebook. `validos = [registro for registro in emails if re.match(padrao, registro["email"])]` é uma **list comprehension** — o mesmo `for` com `if` que você já usou, só que escrito numa linha: "pra cada `registro` em `emails`, guarde só quem passa no `re.match(...)`".

> ⚠️ **Cuidado com padrões "soltos".** Um padrão mais simples como `r".+@.+\..+"` (sem `^`/`$` e sem excluir espaços) parece funcionar, mas na prática aceita coisas erradas — inclusive o e-mail quebrado `"helena martins@grupoalfa.com.br"` (tem espaço) passaria como válido. Teste sempre com um caso que **deveria falhar**, não só com um caso válido.

### ✏️ Atividade 7 — Quem precisa corrigir o e-mail

Usando a lista `invalidos` que você já separou no Exemplo 2, imprima:

a) o **nome** de cada funcionário da lista, pra avisar o time de TI;
b) o **e-mail** (quebrado) de cada funcionário da lista;
c) o **nome e o e-mail juntos**, no formato `nome: email`.

**✅ Gabarito:**

In [ ]:
import json, re

with open("dataset/emails_corporativos.json", encoding="utf-8") as arquivo:
    emails = json.load(arquivo)

padrao = r"^[^\s@]+@[^\s@]+\.[^\s@]+$"
invalidos = [registro for registro in emails if not re.match(padrao, registro["email"])]

# a) só o nome
for registro in invalidos:
    print(registro["funcionario"])

# b) só o e-mail
for registro in invalidos:
    print(registro["email"])

# c) nome e e-mail juntos
for registro in invalidos:
    print(f'{registro["funcionario"]}: {registro["email"]}')

### ✏️ Atividade 8 — Seu próprio e-mail

Monte uma variável com um e-mail no padrão `nome.sobrenome@grupoalfa.com.br` usando o seu nome, e teste se ele passa no padrão.

**✅ Gabarito:**

In [ ]:
import re

padrao = r"^[^\s@]+@[^\s@]+\.[^\s@]+$"
meu_email = "claudio.neves@grupoalfa.com.br"

if re.match(padrao, meu_email):
    print("E-mail parece válido!")
else:
    print("E-mail inválido.")

---
✅ **Checagem rápida — antes de avançar:**
Você consegue calcular quantos minutos de atraso um registro do banco de horas teve, e dizer quantos e-mails do JSON são inválidos?

---

---
## 3. Funções: parâmetros, valores padrão e `return`

> 🔗 **Por que isso agora?** Você já sabe calcular horas trabalhadas e atraso de um registro. Mas se precisar repetir esse cálculo pra outro funcionário — ou pros 100 registros do banco de horas — teria que copiar e colar o código de novo, só trocando os valores. É exatamente esse problema que uma função resolve.

Olhe este código que calcula o atraso de 3 funcionários diferentes, copiado e colado 3 vezes:
```python
atraso_ana = (8*60 + 30) - (8*60)      # 30 min
atraso_bruno = (8*60 + 30) - (8*60)    # 30 min
atraso_carlos = (8*60 + 9) - (8*60)    # 9 min
```
A mesma conta (`entrada_real − entrada_prevista`, em minutos) está escrita 3 vezes — se a tolerância mudar um dia, é fácil esquecer de corrigir uma das linhas. Uma função resolve isso: você escreve a regra **uma única vez**.

### 📖 Antes da função completa: a menor função possível em Python

```python
def dizer_ola():
    return "Olá!"

print(dizer_ola())    # Olá!
```

Repare na anatomia — toda função em Python segue essa mesma receita:
- `def` é a palavra que avisa o Python "a partir daqui eu estou **de**finindo uma função nova" (abreviação de "definir").
- `dizer_ola` é o nome que você escolheu para a função — mesma regra de nomes de variáveis que você já usa.
- `()` é onde ficam os **parâmetros** — "caixinhas vazias" que a função pode pedir pra receber um valor toda vez que for chamada. Aqui está vazio porque essa função não precisa de nenhuma informação de fora.
- A linha indentada (recuada) é o **corpo** da função — o que ela faz de fato.
- `return` é a palavra que diz "é isso que a função devolve pra quem chamou ela".

Agora a mesma receita, com uma "caixinha" (parâmetro) pra preencher:

```python
def dizer_ola_para(nome):
    return f"Olá, {nome}!"

print(dizer_ola_para("Ana"))    # Olá, Ana!
```

É essa mesma receita que a função `calcular_atraso` abaixo usa — só que com 4 caixinhas (parâmetros) em vez de 1, e duas delas já vêm com um valor "pré-preenchido" (o valor padrão) caso você não informe nada.

In [ ]:
def dizer_ola():
    return "Olá!"

print(dizer_ola())

def dizer_ola_para(nome):
    return f"Olá, {nome}!"

print(dizer_ola_para("Ana"))

In [ ]:
def calcular_atraso(hora_entrada, minuto_entrada, hora_prevista=8, minuto_previsto=0):
    minutos_reais = hora_entrada * 60 + minuto_entrada
    minutos_previstos = hora_prevista * 60 + minuto_previsto
    return minutos_reais - minutos_previstos

print(calcular_atraso(8, 30))        # entrada 08:30, previsto padrão 08:00 → 30
print(calcular_atraso(7, 45))        # chegou antes → -15

💡 **"Sem `return`, a função é surda — não responde nada."** Uma função sem `return` pode até imprimir algo na tela, mas não devolve nenhum valor que você possa guardar em uma variável ou usar em outro cálculo.

### 🔹 Exemplo 1 — Função com retorno: horas trabalhadas

In [ ]:
from openpyxl import load_workbook

def calcular_horas_trabalhadas(entrada, saida):
    return (saida - entrada).seconds / 3600

planilha = load_workbook("dataset/recursos_humanos.xlsx")
aba_ponto = planilha["banco_horas"]

registro = None
for linha in aba_ponto.iter_rows(min_row=2, values_only=True):
    if linha[0] == 1 and linha[3].date().isoformat() == "2023-01-02":
        registro = linha  # Ana Souza (id 1), 02/01/2023
        break

entrada = registro[4]
saida = registro[5]
print(f"{calcular_horas_trabalhadas(entrada, saida):.2f}h")

### 🔹 Exemplo 2 — Parâmetro com valor padrão: classificar pontualidade

In [ ]:
def classificar_pontualidade(atraso_minutos, tolerancia=10):
    if atraso_minutos <= tolerancia:
        return "Pontual"
    return "Atrasado"

print(classificar_pontualidade(5))                   # dentro da tolerância padrão: Pontual
print(classificar_pontualidade(30))                  # acima da tolerância padrão: Atrasado
print(classificar_pontualidade(12, tolerancia=15))   # tolerância customizada: Pontual

> ⚠️ **Veja como é um erro real do Python.** Chame `calcular_horas_trabalhadas()` sem passar os dois parâmetros numa célula nova — você vai ver `TypeError: calcular_horas_trabalhadas() missing 2 required positional arguments: 'entrada' and 'saida'`. Diferente de `tolerancia` (que tem valor padrão e pode ser omitido), `entrada` e `saida` não têm padrão — por isso são **obrigatórios**.

### ✏️ Atividade 9 — `calcular_horas_trabalhadas`

Crie a função e teste com 2 registros diferentes do banco de horas.

**✅ Gabarito:**

In [ ]:
from openpyxl import load_workbook

def calcular_horas_trabalhadas(entrada, saida):
    return (saida - entrada).seconds / 3600

planilha = load_workbook("dataset/recursos_humanos.xlsx")
aba_ponto = planilha["banco_horas"]

registro_ana = None
registro_carlos = None
for linha in aba_ponto.iter_rows(min_row=2, values_only=True):
    if linha[3].date().isoformat() == "2023-01-02":
        if linha[0] == 1:
            registro_ana = linha      # Ana Souza
        elif linha[0] == 3:
            registro_carlos = linha   # Carlos Mendes

print(f"Ana Souza: {calcular_horas_trabalhadas(registro_ana[4], registro_ana[5]):.2f}h")
print(f"Carlos Mendes: {calcular_horas_trabalhadas(registro_carlos[4], registro_carlos[5]):.2f}h")

### ✏️ Atividade 10 — `calcular_atraso` com parâmetro padrão

Crie a função `calcular_atraso(hora_entrada, minuto_entrada, hora_prevista=8, minuto_previsto=0)` e teste uma vez com a entrada prevista padrão (08:00) e outra vez com um horário customizado (ex.: turno que começa às 07:00).

**✅ Gabarito:**

In [ ]:
def calcular_atraso(hora_entrada, minuto_entrada, hora_prevista=8, minuto_previsto=0):
    minutos_reais = hora_entrada * 60 + minuto_entrada
    minutos_previstos = hora_prevista * 60 + minuto_previsto
    return minutos_reais - minutos_previstos

# entrada prevista padrão (08:00) — registro de Ana Souza, entrada 08:30
print(calcular_atraso(8, 30))

# turno customizado, começando às 07:00 — mesma entrada real (08:30)
print(calcular_atraso(8, 30, hora_prevista=7, minuto_previsto=0))

### ✏️ Atividade 11 — `validar_email`

Transforme a checagem de regex da Seção 2 em uma função `validar_email(email)` que retorna `True` ou `False`.

**✅ Gabarito:**

In [ ]:
import re

def validar_email(email):
    padrao = r"^[^\s@]+@[^\s@]+\.[^\s@]+$"
    return bool(re.match(padrao, email))

print(validar_email("ana.souza@grupoalfa.com.br"))          # True
print(validar_email("helena martins@grupoalfa.com.br"))     # False

### ✏️ Atividade 12 — `classificar_pontualidade`

Crie a função (como no Exemplo 2) e teste com 3 valores diferentes de atraso.

**✅ Gabarito:**

In [ ]:
def classificar_pontualidade(atraso_minutos, tolerancia=10):
    if atraso_minutos <= tolerancia:
        return "Pontual"
    return "Atrasado"

print(classificar_pontualidade(8))                    # dentro da tolerância padrão: Pontual
print(classificar_pontualidade(20))                   # acima da tolerância padrão: Atrasado
print(classificar_pontualidade(11, tolerancia=12))    # tolerância customizada: Pontual

> 🚀 **Avançado (opcional) — parâmetros "ilimitados" com `*args`:** toda função até aqui tinha um número fixo de parâmetros. Com `*args`, uma função aceita **qualquer quantidade** de argumentos, agrupados numa tupla:
> ```python
> def media(*args):
>     return sum(args) / len(args)
>
> print(media(7, 8, 9))        # funciona com 3 valores
> print(media(7, 8, 9, 10))    # e também com 4, sem mudar a função
> ```
> Isso é a base do desafio opcional ao final desta seção.

---
✅ **Checagem rápida — antes de avançar:**
Você consegue escrever uma função com um parâmetro obrigatório e um parâmetro com valor padrão, e explicar a diferença entre os dois?

---

---
## 4. Lambda e Módulos

> 🔗 **Por que isso agora?** Uma `lambda` não é um conceito novo — é só um jeito mais curto de escrever uma função pequena como as que você acabou de criar. E um módulo é só um arquivo com as suas funções guardadas, prontas pra reusar sem copiar e colar.

### 🔹 Exemplo 1 — Função lambda

In [ ]:
calcular_atraso_lambda = lambda entrada_min, previsto_min=8*60: entrada_min - previsto_min

print(calcular_atraso_lambda(8*60 + 30))    # 30

> ⚠️ **Onde a lambda esbarra.** Tente escrever `lambda x: return x * 2` numa célula nova — o Python recusa com `SyntaxError: invalid syntax`. Lambda só aceita uma **expressão** (algo que produz um valor), nunca um comando como `return`, `if` sozinho ou um `for`. É por isso que lambda serve pra cálculos curtos, não pra lógica complexa.

### 🔹 Exemplo 2 — Onde o lambda realmente brilha: dentro de `min()`/`sorted()`

Guardar uma lambda numa variável (Exemplo 1) é raro na prática. O uso real é passar a lambda **direto** como argumento de outra função, pra dizer "compare por *este* critério":

In [ ]:
import csv
from openpyxl import load_workbook

with open("dataset/cadastro_funcionario.csv", encoding="utf-8") as arquivo:
    funcionarios = list(csv.DictReader(arquivo))
nome_por_id = {int(f["id_funcionario"]): f["funcionario"] for f in funcionarios}

planilha = load_workbook("dataset/recursos_humanos.xlsx")
aba_ponto = planilha["banco_horas"]

ids_do_dia = [1, 2, 8]  # Ana Souza, Bruno Lima, Helena Martins
registros_do_dia = []
for linha in aba_ponto.iter_rows(min_row=2, values_only=True):
    if linha[0] in ids_do_dia and linha[3].date().isoformat() == "2023-01-02":
        registros_do_dia.append({"funcionario": nome_por_id[linha[0]], "entrada": linha[4]})

primeiro_a_chegar = min(registros_do_dia, key=lambda r: r["entrada"])
print(f"Primeiro a chegar: {primeiro_a_chegar['funcionario']}")

### Criando seu próprio módulo

Tanto no Colab quanto no VS Code (rodando este notebook com a extensão Jupyter), criamos o "módulo" como um arquivo de texto usando o comando mágico `%%writefile` — ele funciona nos dois ambientes porque ambos rodam sobre um kernel Jupyter, e cria o arquivo `utilidades_rh.py` na mesma pasta do notebook.

> 📁 **Onde o arquivo é criado?** No Colab, dentro da pasta temporária `/content` (some quando a sessão termina — por isso reenvie os 3 arquivos originais se voltar depois). No VS Code local, na mesma pasta do notebook, como um arquivo `.py` de verdade — permanente, e que você pode (e deve) versionar no Git.

In [ ]:
%%writefile utilidades_rh.py
def calcular_horas_trabalhadas(entrada, saida):
    return (saida - entrada).seconds / 3600

def calcular_atraso(hora_entrada, minuto_entrada, hora_prevista=8, minuto_previsto=0):
    minutos_reais = hora_entrada * 60 + minuto_entrada
    minutos_previstos = hora_prevista * 60 + minuto_previsto
    return minutos_reais - minutos_previstos

def classificar_pontualidade(atraso_minutos, tolerancia=10):
    if atraso_minutos <= tolerancia:
        return "Pontual"
    return "Atrasado" 

In [ ]:
import utilidades_rh

atraso = utilidades_rh.calcular_atraso(8, 30)
print(utilidades_rh.classificar_pontualidade(atraso))

### ✏️ Atividade 13 — Lambda

Reescreva `calcular_horas_trabalhadas` (Seção 3) como uma `lambda`.

**✅ Gabarito:**

In [ ]:
from openpyxl import load_workbook

calcular_horas_trabalhadas_lambda = lambda entrada, saida: (saida - entrada).seconds / 3600

planilha = load_workbook("dataset/recursos_humanos.xlsx")
aba_ponto = planilha["banco_horas"]

registro = None
for linha in aba_ponto.iter_rows(min_row=2, values_only=True):
    if linha[0] == 1 and linha[3].date().isoformat() == "2023-01-02":
        registro = linha  # Ana Souza (id 1), 02/01/2023
        break

entrada = registro[4]
saida = registro[5]
print(f"{calcular_horas_trabalhadas_lambda(entrada, saida):.2f}h")

### ✏️ Atividade 14 — Ordenando por critério

Monte uma lista com 3-4 registros de entrada (pode reaproveitar dados reais do banco de horas) e use `sorted()` com `key=lambda` pra ordenar do que chegou mais cedo ao mais tarde.

**✅ Gabarito:**

In [ ]:
import csv
from openpyxl import load_workbook

with open("dataset/cadastro_funcionario.csv", encoding="utf-8") as arquivo:
    funcionarios = list(csv.DictReader(arquivo))
nome_por_id = {int(f["id_funcionario"]): f["funcionario"] for f in funcionarios}

planilha = load_workbook("dataset/recursos_humanos.xlsx")
aba_ponto = planilha["banco_horas"]

ids_do_dia = [1, 3, 8]  # Ana Souza, Carlos Mendes, Helena Martins
registros = []
for linha in aba_ponto.iter_rows(min_row=2, values_only=True):
    if linha[0] in ids_do_dia and linha[3].date().isoformat() == "2023-01-02":
        registros.append({"funcionario": nome_por_id[linha[0]], "entrada": linha[4]})

ordenado_por_chegada = sorted(registros, key=lambda r: r["entrada"])

for r in ordenado_por_chegada:
    print(r["funcionario"], r["entrada"].strftime("%H:%M"))

### ✏️ Atividade 15 — Adicionar uma função ao módulo

Adicione a função `validar_email` (Atividade 11) ao arquivo `utilidades_rh.py` (reescreva o arquivo com `%%writefile`, incluindo as funções anteriores + a nova).

In [ ]:
%%writefile utilidades_rh.py
import re

def calcular_horas_trabalhadas(entrada, saida):
    return (saida - entrada).seconds / 3600

def calcular_atraso(hora_entrada, minuto_entrada, hora_prevista=8, minuto_previsto=0):
    minutos_reais = hora_entrada * 60 + minuto_entrada
    minutos_previstos = hora_prevista * 60 + minuto_previsto
    return minutos_reais - minutos_previstos

def classificar_pontualidade(atraso_minutos, tolerancia=10):
    if atraso_minutos <= tolerancia:
        return "Pontual"
    return "Atrasado"

def validar_email(email):
    padrao = r"^[^\s@]+@[^\s@]+\.[^\s@]+$"
    return bool(re.match(padrao, email))

### ✏️ Atividade 16 — Usando o módulo reimportado

Reimporte o módulo (com `importlib.reload`, já que ele foi reescrito) e use `validar_email` para processar os e-mails de `emails_corporativos.json`.

**✅ Gabarito:**

In [ ]:
import importlib
import utilidades_rh
importlib.reload(utilidades_rh)  # recarrega o módulo já que ele foi reescrito

import json

with open("dataset/emails_corporativos.json", encoding="utf-8") as arquivo:
    emails = json.load(arquivo)

for registro in emails:
    if not utilidades_rh.validar_email(registro["email"]):
        print(f"E-mail inválido: {registro['funcionario']} ({registro['email']})")

---
✅ **Checagem rápida — antes de avançar:**
Você consegue explicar em uma frase por que uma lambda não pode ter `return` dentro dela?

---

---
### 🏁 Fechamento — Semana 04

**Nesta semana você aprendeu:**
- Leitura e escrita de CSV, JSON e Excel — a base para trabalhar com dados reais de qualquer sistema
- Funções bem definidas: parâmetros, `return`, lambda e módulos
- Datas e expressões regulares para lidar com dados de ponto e cadastro

**Próxima semana:** Semana 05 — Pandas e NumPy: as ferramentas centrais do analista de dados.

---
## 5. Treino em Squads — Sexta-feira (Encontro 3)

Esta seção é usada **em sala (ou em salas remotas/breakout)** na sexta-feira. Cada squad recebe **um setor do Grupo Alfa** (Comercial, Logística, Estoque ou RH) e precisa consolidar as 3 fontes de dados pra responder uma pergunta real de People Analytics:

> **Desafio:** usando os 3 arquivos (`cadastro_funcionario.csv`, `emails_corporativos.json`, `recursos_humanos.xlsx`), filtre só os funcionários do setor do seu squad, descubra **quem teve mais atraso acumulado** na semana de dados disponível, e verifique se o e-mail dessa pessoa está correto. Organizem o código do squad em pelo menos uma função reutilizável.

**✅ Gabarito:**

In [ ]:
# SQUAD — troque "Comercial" pelo setor que seu squad recebeu: Comercial, Logística, Estoque ou RH
import csv, json, re
from openpyxl import load_workbook
from collections import defaultdict

with open("dataset/cadastro_funcionario.csv", encoding="utf-8") as arquivo:
    funcionarios = list(csv.DictReader(arquivo))
setor_por_id = {int(f["id_funcionario"]): f["setor"] for f in funcionarios}
nome_por_id = {int(f["id_funcionario"]): f["funcionario"] for f in funcionarios}

with open("dataset/emails_corporativos.json", encoding="utf-8") as arquivo:
    emails = json.load(arquivo)
email_por_id = {e["id_funcionario"]: e["email"] for e in emails}

padrao = r"^[^\s@]+@[^\s@]+\.[^\s@]+$"

def validar_email(email):
    return bool(re.match(padrao, email))

planilha = load_workbook("dataset/recursos_humanos.xlsx")
aba_ponto = planilha["banco_horas"]

atraso_por_funcionario = defaultdict(int)
for linha in aba_ponto.iter_rows(min_row=2, values_only=True):
    id_func, cargo, setor, data_ref, entrada, saida, jornada = linha
    atraso_min = entrada.hour * 60 + entrada.minute - 8 * 60
    if atraso_min > 0:
        atraso_por_funcionario[id_func] += atraso_min

SETOR_DO_SQUAD = "Comercial"  # troque pelo setor do seu squad

ids_do_setor = [i for i in atraso_por_funcionario if setor_por_id[i] == SETOR_DO_SQUAD]
id_mais_atrasado = max(ids_do_setor, key=lambda i: atraso_por_funcionario[i])

nome = nome_por_id[id_mais_atrasado]
email = email_por_id[id_mais_atrasado]

print(f"Setor: {SETOR_DO_SQUAD}")
print(f"Mais atrasado: {nome} ({atraso_por_funcionario[id_mais_atrasado]} min)")
print(f"E-mail: {email} — {'válido' if validar_email(email) else 'INVÁLIDO'}")

### 🗣️ Debate coletivo (após as apresentações)

Depois que todos os squads apresentarem, discuta com a turma:

- Os squads chegaram ao mesmo resultado pro setor deles?
- Alguém encontrou um funcionário com e-mail inválido?
- As funções ficaram parecidas entre os squads, ou cada um organizou diferente?

### Desafio (opcional)

Reescreva a função `calcular_horas_trabalhadas` usando `*args` pra calcular a média de horas trabalhadas em vários registros de uma vez, sem passar uma lista.

---
### Assinatura

Curso: **Análise de Dados com Python — SENAI (Turma T5)**
Semana 04 — Manipulação de Arquivos e Modularização

*Prof. Especialista Cláudio F. Neves*